In [1]:
#pyg or py data, dataset, dataloader?
# dictionary or Data as output of PROTACDataset?
    #What works with the data loader?
    #What works with pyg funcitons such as num_node_features?



# import torch dataset and dataloader

#from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
# Import from_smiles from pytorch geometric
from torch_geometric.utils import from_smiles
from torch_geometric.data import Data, Dataset, InMemoryDataset
from torch_geometric.loader import DataLoader
from all_functions import get_node_labels
from rdkit import Chem


    

class ProtacDataset(InMemoryDataset):  #Changed from pytorch Dataset to PyG InMemoryDataset

    def __init__(self, protac_df, transform=None):
        self.protac_df = protac_df
        self.protac_smiles = protac_df['PROTAC SMILES'].tolist()
        self.poi_smiles = protac_df['POI SMILES'].tolist()
        self.e3_smiles = protac_df['E3 SMILES'].tolist()
        self.substructures = ['.'.join(protac_df[['POI SMILES', 'LINKER SMILES','E3 SMILES']].iloc[i].tolist()) for i in range(len(protac_df))] #remove later when I have updated functions to not use split_sort. Possible as the substructures will be already separated in the dataframe

        self.node_boundaries = [get_node_labels(
            smiles, substructures_joined) for smiles, substructures_joined in zip(self.protac_smiles, self.substructures)]
    
    def __len__(self):
        return len(self.protac_df)
    
    def __getitem__(self, idx):
       
        #elem = { 'pyg_data': from_smiles(self.protac_smiles[idx]), # (1024,)}

        #elem = {
        #    'x': x ,            
        #    'edge_index': edge_index,
        #    'edge_attr': edge_attr,
        #    'smiles': smiles,
        #    'node_boundaries': self.node_boundaries[idx], # List of boundary classes for each node # (num_nodes,) # boundary_ligand_nodes
        #}
        
        #x, edge_index, edge_features, smiles = from_smiles(self.protac_smiles[idx])
        #elem = Data(
        #        x=x,  #Node features
        #        edge_index=edge_index,
        #        edge_attr=edge_features,
        #        smiles=smiles,
        #        node_boundaries=self.node_boundaries[idx]) 

        elem = from_smiles(self.protac_smiles[idx])
        elem["node_boundaries"] = self.node_boundaries[idx]

        return elem


/home/knkn308/.conda/envs/env-protac-toolkit/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from torch_geometric.nn import GraphConv
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


#optimizer = optim.Adam(model.parameters(), lr=0.001)

#GCNConv worked well before, with a test accuracy of around 0.93
#GraphConv: maybe converges faster?
class PROTACSplitter(torch.nn.Module):
    def __init__(self, node_feature_dim, edge_feature_dim):
        super(PROTACSplitter, self).__init__()
        self.conv1 = GraphConv(node_feature_dim, 16)# edge_dim=edge_feature_dim)
        self.linear_layer = torch.nn.Linear(16, 3)  # Output layer for 3 classes
    
    #def forward(self, data_batch):
    def forward(self, node_attr, edge_index):
        #data = batch['pyg_data']
        #print(f"data_batch.x: {data_batch.x}")
        #print(f"data_batch.edge_index: {data_batch.edge_index}")
        #z = self.conv1(data_batch.x, data_batch.edge_index)#, edge_attr)
        z = self.conv1(node_attr, edge_index)#, edge_attr)
        z = F.relu(z)
        y = self.linear_layer(z)
        return y
    
    def train_model(self, training_data, optimizer=optim.Adam, lr = 0.001, batch_size=32, criterion=torch.nn.CrossEntropyLoss, shuffle=True):
        self.train()
        #training_pyg_data = training_data['pyg_data']
        train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=shuffle)
        total_loss = 0
        #print(train_loader)
        for train_data in train_loader:
            optimizer=optimizer(self.parameters(), lr=lr)
            optimizer.zero_grad()
            node_attr=train_data.x
            edge_index = train_data.edge_index
            #print(type(node_attr))
            print(node_attr[0][0])
            print(edge_index[0][0])
            raw_boundary_prediction = self.forward(node_attr, edge_index)             # "RuntimeError: mat1 and mat2 must have the same dtype"
            node_class_targets = train_data['node_boundaries']
            loss = criterion(raw_boundary_prediction, node_class_targets) 
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / batch_size

        return avg_loss

In [2]:
protac_pub_trainset_df = pd.read_csv('../../data/augmented/protac_pub_testset_testing.csv')
protac_pub_testset_df = pd.read_csv('../../data/augmented/protac_pub_trainset_testing.csv')
train_set_pub = ProtacDataset(protac_df=protac_pub_trainset_df)

In [33]:
#node_feature_dim = train_set_pub[0]['x'][1].num_node_features
node_feature_dim = train_set_pub.num_node_features
edge_feature_dim = train_set_pub.num_edge_features 
model = PROTACSplitter(node_feature_dim, edge_feature_dim)

In [34]:
model.train_model(training_data=train_set_pub, batch_size=32)

tensor(6)
tensor(0)


RuntimeError: mat1 and mat2 must have the same dtype

In [36]:
from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_set_pub, batch_size=32, shuffle=True)
x = next(iter(train_loader))

In [ ]:


model = PROTACSplitter()
for batch in DataLoader(train_set_pub, batch_size=32):
    # batch['protac_smiles'] = (batch_size, 1024)
    y_hat = model(batch)
    loss = loss_fn(y_hat, batch['node_boundaries'])
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Metric
    y_hat = torch.argmax(y_hat, dim=1) # (batch_size, num_nodes) -> (batch_size, 1)
    acc = (y_hat == batch['node_boundaries']).sum() / len(y_hat)
